In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tabulate import tabulate
import itertools

# Φόρτωση όλων των μετοχών
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

symbols = ["AAPL","AIG", "BAC", "GS","MNST","MSFT","NKE","PFE","PLUG","SHOP","SPCE"]
path = "/content/drive/MyDrive/Βαση Δεδομενων/"

datasets = []

for symbol in symbols:
    df = pd.read_excel(f"{path}{symbol}_Data_Technical_Indicators_Combined.xlsx", parse_dates=[0])
    df.rename(columns={df.columns[0]: "Date"}, inplace=True)
    df.set_index("Date", inplace=True)
    df = df.sort_index()
    # Δημιουργία Next_Volume
    df["Next_Volume"] = df.iloc[:, 4].shift(-1)
    df["Symbol"] = symbol
    datasets.append(df)

# Κοινές ημερομηνίες για όλες τις μετοχές
common_dates = datasets[0].index
for df in datasets[1:]:
    common_dates = common_dates.intersection(df.index)

# Φιλτράρουμε όλα τα dataframes για τις κοινές ημερομηνίες
for i in range(len(datasets)):
    datasets[i] = datasets[i].loc[common_dates]

# Ενοποίηση όλων των datasets
dataset = pd.concat(datasets, axis=0)
dataset = dataset.dropna()

le = LabelEncoder()
dataset["Symbol_enc"] = le.fit_transform(dataset["Symbol"])
num_symbols = dataset["Symbol_enc"].nunique()

# Ορισμός μεταβλητών
required_columns = [5, 8, 9, 10, 11, 14]
# Προαιρετικές στήλες που θα δοκιμάζονται σε συνδυασμούς
optional_columns = [3, 15, 16, 17, 18, 20]


# Δημιουργία συνδυασμών
column_sets = []
for r in range(0, len(optional_columns) + 1):
    for combo in itertools.combinations(optional_columns, r):
        columns = required_columns + list(combo)
        if len(columns) >= 5:  # τουλάχιστον 5 features
            column_sets.append(columns)

print(f"Σύνολο συνδυασμών που θα δοκιμαστούν: {len(column_sets)}")

target_column = "Next_Volume"
test_size = 0.2


# Εκπαίδευση και αξιολόγηση
results = []

for i, columns in enumerate(column_sets, start=1):
    X_num = dataset.iloc[:, columns]
    X_sym = dataset["Symbol_enc"]
    y = dataset[target_column]

    dates = dataset.index

    valid_idx = y.notna()
    X_num = X_num.loc[valid_idx]
    X_sym = X_sym.loc[valid_idx]
    y = np.log1p(y.loc[valid_idx])

    total_len = len(common_dates)
    test_len = int(total_len * test_size)

    train_dates = common_dates[:-test_len]
    test_dates  = common_dates[-test_len:]

    X_num_train = X_num.loc[X_num.index.isin(train_dates)]
    X_sym_train = X_sym.loc[X_sym.index.isin(train_dates)]
    y_train = y.loc[y.index.isin(train_dates)]

    X_num_test = X_num.loc[X_num.index.isin(test_dates)]
    X_sym_test = X_sym.loc[X_sym.index.isin(test_dates)]
    y_test = y.loc[y.index.isin(test_dates)]

    val_size = 0.2
    val_len = int(len(train_dates) * val_size)

    train_dates_final = train_dates[:-val_len]
    val_dates = train_dates[-val_len:]

    X_num_train_final = X_num.loc[X_num.index.isin(train_dates_final)]
    X_sym_train_final = X_sym.loc[X_sym.index.isin(train_dates_final)]
    y_train_final = y.loc[y.index.isin(train_dates_final)]

    X_num_val = X_num.loc[X_num.index.isin(val_dates)]
    X_sym_val = X_sym.loc[X_sym.index.isin(val_dates)]
    y_val = y.loc[y.index.isin(val_dates)]

    sc = StandardScaler()
    X_num_train_final = sc.fit_transform(X_num_train_final)
    X_num_val = sc.transform(X_num_val)
    X_num_test = sc.transform(X_num_test)


    # Νευρωνικό Δίκτυο
    num_input = Input(shape=(X_num_train_final.shape[1],), name="num_input")
    sym_input = Input(shape=(1,), name="sym_input")

    embed_dim = 2
    sym_embed = Embedding(input_dim=num_symbols,output_dim=embed_dim,name="symbol_embedding")(sym_input)
    sym_embed = Flatten()(sym_embed)
    x = Concatenate()([num_input, sym_embed])
    x = Dense(32, activation="relu")(x)
    x = Dense(16, activation="relu")(x)
    output = Dense(1, activation="linear")(x)
    model = Model(inputs=[num_input, sym_input], outputs=output)
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae', 'mse'])
    early_stopping = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    history=model.fit([X_num_train_final, X_sym_train_final],y_train_final,validation_data=([X_num_val, X_sym_val], y_val),batch_size = 16, epochs = 1000,callbacks=[early_stopping], verbose=0)

    train_loss, train_mae, train_mse = model.evaluate([X_num_train_final, X_sym_train_final], y_train_final, verbose=0)
    val_loss, val_mae, val_mse = model.evaluate([X_num_val, X_sym_val], y_val, verbose=0)

    results.append({
        'Set': i,
        'Columns': columns,
        'Train MAE': train_mae,
        'Val MAE': val_mae,
        'Train MSE': train_mse,
        'Val MSE': val_mse
    })

    print(f"[{i}/{len(column_sets)}] Ολοκληρώθηκε -> Val MAE: {val_mae:.4f}")


# Εμφάνιση & αποθήκευση αποτελεσμάτων
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Val MAE', ascending=True)

print("\nΚαλύτερα 5 αποτελέσματα:")
print(tabulate(results_df.head(5), headers='keys', tablefmt='grid'))

# Αποθήκευση σε αρχείο CSV
results_df.to_csv("/content/drive/MyDrive/Best_Feature_Next_Volume_Combinations_Results.csv", index=False)
results_df.to_excel("/content/drive/MyDrive/Best_Feature_Next_Volume_Combinations_Results.xlsx", index=False)
print("\nΤα αποτελέσματα αποθηκεύτηκαν στο Google Drive ")


Mounted at /content/drive
Σύνολο συνδυασμών που θα δοκιμαστούν: 64
[1/64] Ολοκληρώθηκε -> Val MAE: 0.3372
[2/64] Ολοκληρώθηκε -> Val MAE: 0.3763
[3/64] Ολοκληρώθηκε -> Val MAE: 0.3138
[4/64] Ολοκληρώθηκε -> Val MAE: 0.3480
[5/64] Ολοκληρώθηκε -> Val MAE: 0.3229
[6/64] Ολοκληρώθηκε -> Val MAE: 0.3518
[7/64] Ολοκληρώθηκε -> Val MAE: 0.3218
[8/64] Ολοκληρώθηκε -> Val MAE: 0.3044
[9/64] Ολοκληρώθηκε -> Val MAE: 0.3369
[10/64] Ολοκληρώθηκε -> Val MAE: 0.3449
[11/64] Ολοκληρώθηκε -> Val MAE: 0.3307
[12/64] Ολοκληρώθηκε -> Val MAE: 0.3180
[13/64] Ολοκληρώθηκε -> Val MAE: 0.3150
[14/64] Ολοκληρώθηκε -> Val MAE: 0.3041
[15/64] Ολοκληρώθηκε -> Val MAE: 0.3151
[16/64] Ολοκληρώθηκε -> Val MAE: 0.3302
[17/64] Ολοκληρώθηκε -> Val MAE: 0.3192
[18/64] Ολοκληρώθηκε -> Val MAE: 0.3760
[19/64] Ολοκληρώθηκε -> Val MAE: 0.3129
[20/64] Ολοκληρώθηκε -> Val MAE: 0.3159
[21/64] Ολοκληρώθηκε -> Val MAE: 0.3311
[22/64] Ολοκληρώθηκε -> Val MAE: 0.3359
[23/64] Ολοκληρώθηκε -> Val MAE: 0.3354
[24/64] Ολοκληρώθηκε -